In [55]:
from pathlib import Path
import pandas as pd
import cv2
import yaml
from typing import Tuple
from tqdm.notebook import tqdm
from ultralytics import YOLO

In [56]:
TEST_CSV_PATH = Path("./data/DeepFish/Segmentation/test.csv")
TRAIN_CSV_PATH = Path("./data/DeepFish/Segmentation/train.csv")
VAL_CSV_PATH = Path("./data/DeepFish/Segmentation/val.csv")

TEST_CSV_PATH.exists(), TRAIN_CSV_PATH.exists(), VAL_CSV_PATH.exists()

(True, True, True)

In [57]:
test_df = pd.read_csv(TEST_CSV_PATH)
train_df = pd.read_csv(TRAIN_CSV_PATH)
val_df = pd.read_csv(VAL_CSV_PATH)

In [58]:
IMAGE_DIR = Path("../datasets/DeepFish/Segmentation/images")
IMAGE_TRAIN_DIR = IMAGE_DIR / "train"
IMAGE_VAL_DIR = IMAGE_DIR / "val"
IMAGE_TEST_DIR = IMAGE_DIR / "test"

IMAGE_DIR.mkdir(parents=True, exist_ok=True)
IMAGE_TRAIN_DIR.mkdir(parents=True, exist_ok=True)
IMAGE_VAL_DIR.mkdir(parents=True, exist_ok=True)
IMAGE_TEST_DIR.mkdir(parents=True, exist_ok=True)

LABEL_DIR = Path("../datasets/DeepFish/Segmentation/labels")
LABEL_TRAIN_DIR = LABEL_DIR / "train"
LABEL_VAL_DIR = LABEL_DIR / "val"
LABEL_TEST_DIR = LABEL_DIR / "test"

LABEL_DIR.mkdir(parents=True, exist_ok=True)
LABEL_TRAIN_DIR.mkdir(parents=True, exist_ok=True)
LABEL_VAL_DIR.mkdir(parents=True, exist_ok=True)
LABEL_TEST_DIR.mkdir(parents=True, exist_ok=True)

In [59]:
SEGMENTATION_IMAGES = list(Path("./data/DeepFish/Segmentation/images").glob("*/*.jpg"))
SEGMENTATION_MASKS = list(Path("./data/DeepFish/Segmentation/masks").glob("*/*.png"))

len(SEGMENTATION_IMAGES), len(SEGMENTATION_MASKS)

(620, 620)

In [60]:
segmentation_mask_map = {mask.stem: mask for mask in SEGMENTATION_MASKS}
segmentation_image_mask_pairs = [(image, segmentation_mask_map[image.stem]) for image in SEGMENTATION_IMAGES if image.stem in segmentation_mask_map]

len(segmentation_image_mask_pairs)

620

In [61]:
def mask_to_polygon(segmentation_mask: Path):
    # Read mask as single-channel and binarize to ensure valid contours
    mask = cv2.imread(str(segmentation_mask), cv2.IMREAD_UNCHANGED)
    if mask is None:
        return []
    if len(mask.shape) > 2:
        mask = cv2.cvtColor(mask, cv2.COLOR_BGR2GRAY)
    _, thresh = cv2.threshold(mask, 127, 255, cv2.THRESH_BINARY)
    contours, _ = cv2.findContours(thresh, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

    polygons = []
    for contour in contours:
        polygon = contour.reshape(-1, 2)
        if len(polygon) >= 3:
            polygons.append(polygon.tolist())

    return polygons


In [62]:
image_polygon_pairs = [(image, mask_to_polygon(mask)) for image, mask in segmentation_image_mask_pairs]

In [63]:
# point to the actual dataset location (datasets was moved up one level)
deepfish_root = Path("../datasets/DeepFish").resolve()
deepfish_yaml = {
    "path": str(deepfish_root),
    "train": "Segmentation/images/train",
    "val": "Segmentation/images/val",
    "test": "Segmentation/images/test",
    "names": ["fish"],
}

# write YAML at repo root so ultralytics resolves absolute paths reliably
with open("./deepfish_segmentation.yaml", "w") as f:
    yaml.dump(deepfish_yaml, f)

In [64]:
def get_dir_for_image(image: Path) -> Tuple[Path, Path]:
    image_id = image.stem
    if image_id in {value.split("/")[-1] for value in test_df["ID"].values}:
        return IMAGE_TEST_DIR, LABEL_TEST_DIR
    elif image_id in {value.split("/")[-1] for value in train_df["ID"].values}:
        return IMAGE_TRAIN_DIR, LABEL_TRAIN_DIR
    elif image_id in {value.split("/")[-1] for value in val_df["ID"].values}:
        return IMAGE_VAL_DIR, LABEL_VAL_DIR
    else:
        raise ValueError(f"Image {image} not found in any of the CSV files.")

In [65]:
for image, polygons in tqdm(image_polygon_pairs):
    image_name = image.name
    image_path, label_path = get_dir_for_image(image)
    image_path = image_path / image_name
    image_path.parent.mkdir(parents=True, exist_ok=True)
    if not image_path.exists():
        image_path.symlink_to(image.resolve())

    image_array = cv2.imread(str(image_path))
    if image_array is None:
        continue
    img_height, img_width = image_array.shape[:2]

    label_path = label_path / f"{image.stem}.txt"
    # If there are no polygons for this image, remove existing label file and skip
    if not polygons:
        if label_path.exists():
            label_path.unlink()
        continue

    with open(label_path, "w") as f:
        for polygon in polygons:
            coords = []
            for point in polygon:
                x, y = point
                x_norm = float(x) / float(img_width)
                y_norm = float(y) / float(img_height)
                coords.append(f"{x_norm:.6f} {y_norm:.6f}")

            f.write("0 " + " ".join(coords) + "\n")


  0%|          | 0/620 [00:00<?, ?it/s]

In [66]:
model = YOLO("yolo26n-seg.pt")

In [67]:
results = model.train(data="deepfish_segmentation.yaml", epochs=100, imgsz=640)

Ultralytics 8.4.19 🚀 Python-3.13.9 torch-2.10.0+cu128 CUDA:0 (NVIDIA GeForce RTX 3060 Laptop GPU, 5804MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=deepfish_segmentation.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo26n-seg.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=train, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask=True,